# Miniconda and FastGen Environment Setup

This notebook documents the steps to verify conda, install Miniconda if needed, create the `fastgen` environment, and install FastGen.

## 1. Verify Python, pip, and conda availability

In [ ]:
!which python3 && which pip && conda --version 2>&1 || echo "conda not found"

## 2. Install Miniconda (only if conda is not already installed)

If `conda` is missing, run the commands below to download and install Miniconda. This step is optional once Miniconda is already installed.

In [ ]:
# Optional: install Miniconda if conda is not available
# !curl -sL https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -o /tmp/miniconda.sh
# !bash /tmp/miniconda.sh -b -p $HOME/miniconda3
# !bash -lc "source $HOME/.bashrc && conda --version"

## 3. Create the `fastgen` conda environment

In [ ]:
%%bash
# Create the `fastgen` conda env. Idempotent: removes any existing env first so
# re-running this notebook always gives a clean, reproducible install.
source ~/miniconda3/etc/profile.d/conda.sh
# Accept Anaconda channel Terms of Service (required by newer conda for non-interactive use)
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true
conda env remove -n fastgen -y 2>/dev/null || true
conda create -y -n fastgen python=3.12.3 pip

In [ ]:
%%bash
# IMPORTANT: every `%%bash` / `!` cell is a fresh, non-interactive shell.
# `~/.bashrc` no-ops in non-interactive shells, so `conda activate` fails there.
# Source conda's hook and activate the env in EVERY cell that needs it.
source ~/miniconda3/etc/profile.d/conda.sh
conda activate fastgen
python --version

In [ ]:
%%bash
# Activate the env first so `pip` is the env's pip (NOT the system, externally-managed pip).
source ~/miniconda3/etc/profile.d/conda.sh
conda activate fastgen
if [ -d .git ]; then
  echo "Installing from current FastGen repository"
  pip install -e .
else
  git clone https://github.com/NVlabs/FastGen.git repo_clone
  cd repo_clone
  pip install -e .
fi
# pandas is imported by fastgen/callbacks/gpu_stats.py but not declared in install_requires.
pip install pandas
# Register this env as a Jupyter kernel so normal (non-bash) Python cells can use it.
python -m ipykernel install --user --name fastgen --display-name "Python (fastgen GPU)"

In [ ]:
%%bash
# === Verify the installation ===
# Confirms the key packages import from the fastgen env AND the GPU is visible.
# A correct setup prints "RESULT: PASS".
source ~/miniconda3/etc/profile.d/conda.sh
conda activate fastgen
python - <<'PY'
import importlib, sys
print("python:", sys.executable)   # should end in /envs/fastgen/bin/python
ok = True
for m in ["torch", "diffusers", "transformers", "fastgen"]:
    try:
        mod = importlib.import_module(m)
        print(f"  {m}: OK {getattr(mod, '__version__', '')}")
    except Exception as e:
        ok = False
        print(f"  {m}: FAIL -> {e}")
import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
print("RESULT:", "PASS" if (ok and torch.cuda.is_available()) else "FAIL")
PY

## 6. Credentials (Optional)

For W&B logging, [get your API key](https://wandb.ai/settings) and save it to `credentials/wandb_api.txt` or set the `WANDB_API_KEY` environment variable.
Without either of these, W&B will prompt for your API key interactively.

For more details, including S3 storage and other environment variables, see [fastgen/configs/README.md](fastgen/configs/README.md#environment-variables).

In [ ]:
%%bash
source ~/miniconda3/etc/profile.d/conda.sh
conda activate fastgen
python scripts/download_data.py --dataset cifar10

## 7. HF Login

The CosmosPredict2 DMD2 training uses a pretrained **teacher** checkpoint that must be downloaded from a **gated** HuggingFace repo. Before running the cell below:

**Accept the license** at https://huggingface.co/nvidia/Cosmos-Predict2.5-2B (while logged into your HF account).

Then, **Authenticate once** — set your HuggingFace token below (get one at https://huggingface.co/settings/tokens). Don't hard-code the token into the notebook.

In [ ]:
from getpass import getpass
!source ~/miniconda3/etc/profile.d/conda.sh && conda activate fastgen && hf auth login --token "{getpass(prompt='HF_TOKEN:')}" 

### Basic Training

If you run out-of-memory, try a smaller batch-size, e.g., `dataloader_train.batch_size=32`, which automatically uses gradient accumulation to match the global batch-size.

**Expected Output:** See the training log for a link to the run on [wandb.ai](https://wandb.ai). Training outputs go to `$FASTGEN_OUTPUT_ROOT/{project}/{group}/{name}/`. With default settings, outputs are organized as follows:
```
FASTGEN_OUTPUT/fastgen/cifar10/debug/
├── checkpoints/    # Model checkpoints in the format {iteration:07d}.pth
│   ├── 0001000.pth
│   └── ...
├── config.yaml     # Resolved configuration for reproducibility
├── wandb_id.txt    # W&B run ID for resuming
└── ...          
```

In [ ]:
%%bash
source ~/miniconda3/etc/profile.d/conda.sh
conda activate fastgen
python train.py --config=fastgen/configs/experiments/CosmosPredict2/config_dmd2.py - log_config.name=cosmos_predict2_dmd2

In [ ]:
%%bash
source ~/miniconda3/etc/profile.d/conda.sh
conda activate fastgen
python scripts/inference/image_model_inference.py --config fastgen/configs/experiments/EDM/config_dmd2_test.py \
  --classes=10 --prompt_file=scripts/inference/prompts/classes.txt --ckpt_path=FASTGEN_OUTPUT/fastgen/cifar10/debug/checkpoints/0002000.pth - log_config.name=test_inference

In [ ]:
%%bash
source ~/miniconda3/etc/profile.d/conda.sh
conda activate fastgen
python scripts/fid/compute_fid_from_ckpts.py \
    --config fastgen/configs/experiments/EDM/config_dmd2_cifar10.py

## 8. Documentation

Detailed documentation is available in each component's README:

| Component | Documentation | Description |
|-----------|---------------|-------------|
| **Methods** | [fastgen/methods/README.md](fastgen/methods/README.md) | Training methods (sCM, MeanFlow, DMD2, Self-Forcing, etc.) |
| **Networks** | [fastgen/networks/README.md](fastgen/networks/README.md) | Network architectures (EDM, SD, SDXL, Flux, Qwen-Image, WAN, CogVideoX, Cosmos) and pretrained models |
| **Configs** | [fastgen/configs/README.md](fastgen/configs/README.md) | Configuration system, environment variables, and creating custom configs |
| **Datasets** | [fastgen/datasets/README.md](fastgen/datasets/README.md) | Dataset preparation and WebDataset loaders |
| **Callbacks** | [fastgen/callbacks/README.md](fastgen/callbacks/README.md) | Training callbacks (EMA, logging, gradient clipping, etc.) |
| **Inference** | [scripts/README.md](scripts/README.md) | Inference modes (T2I, T2V, I2V, V2V, etc.) and FID evaluation |
| **Third Party** | [fastgen/third_party/README.md](fastgen/third_party/README.md) | Third-party dependencies (Depth Anything V2, etc.) |